In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

project_id = 'data-analytics-mate'
client = bigquery.Client(project=project_id)

In [ ]:
SQL_QUERY = """
WITH session_info AS (
  SELECT
    s.date,
    s.ga_session_id,
    sp.country,
    sp.device,
    sp.continent,
    sp.channel,
    ab.test,
    ab.test_group
  FROM `DA.ab_test` ab
  JOIN `DA.session` s
  ON ab.ga_session_id = s.ga_session_id
  JOIN `DA.session_params` sp
  ON sp.ga_session_id = ab.ga_session_id
),
session_with_orders AS(
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    COUNT(DISTINCT o.ga_session_id) AS session_with_orders
  FROM `DA.order` o
  JOIN session_info
  ON o.ga_session_id = session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),
events AS(
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    sp.event_name,
    COUNT(sp.ga_session_id) AS event_cnt
  FROM `DA.event_params` sp
  JOIN session_info
  ON sp.ga_session_id = session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    sp.event_name
),
session AS(
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    COUNT(DISTINCT session_info.ga_session_id) AS session_cnt
  FROM session_info
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
),
account AS(
  SELECT
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group,
    COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
  FROM `DA.account_session` acs
  JOIN session_info
  ON acs.ga_session_id = session_info.ga_session_id
  GROUP BY
    session_info.date,
    session_info.country,
    session_info.device,
    session_info.continent,
    session_info.channel,
    session_info.test,
    session_info.test_group
)

SELECT
    session_with_orders.date, session_with_orders.country, session_with_orders.device,
    session_with_orders.continent, session_with_orders.channel, session_with_orders.test,
    session_with_orders.test_group, 'session_with_orders' AS event_name,
    session_with_orders.session_with_orders AS value
FROM session_with_orders
UNION ALL
SELECT
    events.date, events.country, events.device, events.continent, events.channel,
    events.test, events.test_group, event_name, event_cnt AS value
FROM events
UNION ALL
SELECT
    session.date, session.country, session.device, session.continent, session.channel,
    session.test, session.test_group, 'session' AS event_name, session_cnt AS value
FROM session
UNION ALL
SELECT
    account.date, account.country, account.device, account.continent, account.channel,
    account.test, account.test_group, 'new account' AS event_name, new_account_cnt AS value
FROM account
"""

df = client.query(SQL_QUERY).to_dataframe(create_bqstorage_client=False)

print(f"Завантажено рядків: {len(df)}")  # має бути ~800 996

# Приводимо назви колонок до нижнього регістру про всяк випадок
df.columns = [c.strip().lower() for c in df.columns]

df.to_csv("ab_test_raw_data.csv", index=False)


Завантажено рядків: 800996


In [ ]:
METRICS = [
    {
        "metric": "add_payment_info / session",
        "numerator_event": "add_payment_info",
        "denominator_event": "session",
    },
    {
        "metric": "add_shipping_info / session",
        "numerator_event": "add_shipping_info",
        "denominator_event": "session",
    },
    {
        "metric": "begin_checkout / session",
        "numerator_event": "begin_checkout",
        "denominator_event": "session",
    },
    {
        "metric": "new_accounts / session",
        "numerator_event": "new account",
        "denominator_event": "session",
    },
]

ALPHA = 0.05  # рівень значущості

In [ ]:
# 3. Функція z-тесту для різниці двох пропорцій
# ------------------------------------------------------------
def z_test_proportions(x1, n1, x2, n2):
    """
    x1, n1 - numerator/denominator групи A (test_group == 1, контроль)
    x2, n2 - numerator/denominator групи B (test_group == 2, тест)
    Повертає (z_stat, p_value)
    """
    if n1 == 0 or n2 == 0:
        return np.nan, np.nan

    p1 = x1 / n1
    p2 = x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    p_pool = min(max(p_pool, 0.0), 1.0)  # захист від похибки округлення

    variance = p_pool * (1 - p_pool) * (1 / n1 + 1 / n2)
    if variance <= 0:
        return np.nan, np.nan
    se = np.sqrt(variance)

    z = (p2 - p1) / se
    p_value = 2 * (1 - norm.cdf(abs(z)))
    return z, p_value

In [ ]:
# 4. Основна функція розрахунку значущості
# ------------------------------------------------------------
def calc_significance(data: pd.DataFrame, slice_cols=None) -> pd.DataFrame:
    slice_cols = slice_cols or []
    group_cols = ["test"] + slice_cols

    results = []

    for keys, group in data.groupby(group_cols):
        keys = keys if isinstance(keys, tuple) else (keys,)
        test_number = keys[0]
        slice_values = keys[1:]

        # Групи А і Б всередині цього зрізу
        group_a = group[group["test_group"] == 1]
        group_b = group[group["test_group"] == 2]

        for m in METRICS:
            num_event = m["numerator_event"]
            den_event = m["denominator_event"]

            numerator_a = group_a.loc[group_a["event_name"] == num_event, "value"].sum()
            denominator_a = group_a.loc[group_a["event_name"] == den_event, "value"].sum()

            numerator_b = group_b.loc[group_b["event_name"] == num_event, "value"].sum()
            denominator_b = group_b.loc[group_b["event_name"] == den_event, "value"].sum()

            # Пропускаємо зрізи, де немає даних для порівняння
            if denominator_a == 0 or denominator_b == 0:
                continue

            conversion_a = numerator_a / denominator_a
            conversion_b = numerator_b / denominator_b

            metric_change_pct = (
                (conversion_b - conversion_a) / conversion_a * 100
                if conversion_a != 0 else np.nan
            )

            z_stat, p_value = z_test_proportions(
                numerator_a, denominator_a, numerator_b, denominator_b
            )

            row = {"test_number": test_number, "metric": m["metric"]}

            # Якщо рахуємо в розрізі (country/device/...) — додаємо ці колонки
            for i, col in enumerate(slice_cols):
                row[col] = slice_values[i]

            row.update({
                "numerator_event": num_event,
                "denominator_event": den_event,
                "numerator_count_a": numerator_a,
                "denominator_count_a": denominator_a,
                "conversion_rate_a": conversion_a,
                "numerator_count_b": numerator_b,
                "denominator_count_b": denominator_b,
                "conversion_rate_b": conversion_b,
                "metric_change_%": metric_change_pct,
                "z_stat": z_stat,
                "p_value": p_value,
                "significant": bool(p_value < ALPHA) if not np.isnan(p_value) else False,
            })

            results.append(row)

    return pd.DataFrame(results)

In [ ]:
# 5. Розрахунок в тоталі по тесту
# ------------------------------------------------------------
results_total = calc_significance(df)
results_total.to_csv("ab_test_results_total.csv", index=False)
print("Total results:")
print(results_total)

# ------------------------------------------------------------
# 6. Розрахунок в усіх можливих розрізах.
# ------------------------------------------------------------
SLICE_DIMENSIONS = ["country", "device", "continent", "channel"]

all_slice_results = []
for dim in SLICE_DIMENSIONS:
    if dim in df.columns:
        res = calc_significance(df, slice_cols=[dim])
        res["slice_dimension"] = dim
        res = res.rename(columns={dim: "slice_value"})
        all_slice_results.append(res)

if all_slice_results:
    results_by_slices = pd.concat(all_slice_results, ignore_index=True)
    results_by_slices.to_csv("ab_test_results_by_slices.csv", index=False)
    print("\nResults by slices (sample):")
    print(results_by_slices.head(20))

Total results:
    test_number                       metric    numerator_event  \
0             1   add_payment_info / session   add_payment_info   
1             1  add_shipping_info / session  add_shipping_info   
2             1     begin_checkout / session     begin_checkout   
3             1       new_accounts / session        new account   
4             2   add_payment_info / session   add_payment_info   
5             2  add_shipping_info / session  add_shipping_info   
6             2     begin_checkout / session     begin_checkout   
7             2       new_accounts / session        new account   
8             3   add_payment_info / session   add_payment_info   
9             3  add_shipping_info / session  add_shipping_info   
10            3     begin_checkout / session     begin_checkout   
11            3       new_accounts / session        new account   
12            4   add_payment_info / session   add_payment_info   
13            4  add_shipping_info / session  a

Посилання на перероблений [Create Your A/B Testing Tool](https://public.tableau.com/shared/Z96YP83P2?:display_count=n&:origin=viz_share_link)


Посилання на [Significance дашборд](https://public.tableau.com/views/PortfolioProject22/Significance?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)


Посилання на [cvs](https://drive.google.com/file/d/10hv4pD1f_6E2Uqoo74zapyfm9UE5Q3UQ/view?usp=sharing)